In [79]:
import pypsa
import pandas as pd
import numpy as np

Parse the datasets from entsoe for total load, pv generation and energy prices (selected date is 15.07.2025)

In [80]:
prices_csv = pd.read_csv("entsoe_datasets/energy_prices_15_07_2025.csv",
                         parse_dates=["time"],
                         index_col=["time"],
                         usecols=lambda col: "time" in col or "Sequence" in col or "Day-ahead" in col,
                         date_format="%Y-%m-%d %H:%M:%S")
prices_csv = prices_csv[prices_csv["Sequence"] == "Sequence 1"]
price_series = []
for price in prices_csv.values:
    price_series.append(price[1])

pv_generation_csv = pd.read_csv("entsoe_datasets/pv_generation_15_07_2025.csv",
                            parse_dates=["time"],
                            index_col=["time"],
                            usecols=lambda col: "time" in col or "Day-ahead" in col,
                            date_format="%Y-%m-%d %H:%M:%S")
normalized_pv = pv_generation_csv.values #/ 32
normalized_pv = normalized_pv / normalized_pv.max()
# (pv_generation_csv.values).max()
pv_series = []
for pv in normalized_pv:
    pv_series.append(pv[0]) # kw

load_csv = pd.read_csv("entsoe_datasets/load_15_07_2025.csv",
                         parse_dates=["time"],
                         index_col=["time"],
                         usecols=lambda col: "time" in col or "Sequence" in col or "Day-ahead" in col,
                         date_format="%Y-%m-%d %H:%M:%S")
normalized_load = load_csv.values / 33
normalized_load = normalized_load / normalized_load.max()
load_series = []
for load in normalized_load:
    load_series.append(load[0]) # kw

Create a PyPSA network and set the snapshots for a period of 24 h.

In [81]:
network = pypsa.Network()
start_time = "2025-07-15 00:00:00"
timestamps = pd.date_range(start_time, periods=24, freq="h")
network.set_snapshots(timestamps)

Add the buses for the IEEE 33-bus system. The base voltage is 12.66 kV

In [82]:
for i in range(1, 34):
    network.add(
        "Bus",
        f"Bus_{i}",
        v_nom=12.66
    )

Adding the lines. Resistance (r) and reactance (x) are in Ohm. The thermal capacity (s_nom) is set high so as to not be a limiting factor

In [83]:
# (from, to, r, x)
line_data = [
    (1, 2, 0.0922, 0.047),      (2, 3, 0.493, 0.2511),      (3, 4, 0.366, 0.1864), 
    (4, 5, 0.3811, 0.1941),     (5, 6, 0.819, 0.707),       (6, 7, 0.1872, 0.6188), 
    (7, 8, 0.7114, 0.2351),     (8, 9, 1.03, 0.74),         (9, 10, 1.044, 0.74),   
    (10, 11, 0.1966, 0.065),    (11, 12, 0.3744, 0.198),    (12, 13, 1.468, 1.155),
    (13, 14, 0.5416, 0.7129),   (14, 15, 0.591, 0.526),     (15, 16, 0.7463, 0.545),
    (16, 17, 1.289, 1.721),     (17, 18, 0.732, 0.574),     (2, 19, 0.164, 0.1565),
    (19, 20, 1.5042, 1.3554),   (20, 21, 0.4095, 0.4784),   (21, 22, 0.7089, 0.9373),   
    (3, 23, 0.4512, 0.3083),    (23, 24, 0.898, 0.7091),    (24, 25, 0.896, 0.7011), 
    (6, 26, 0.203, 0.1034),     (26, 27, 0.2842, 0.1447),   (27, 28, 1.059, 0.9337), 
    (28, 29, 0.8042, 0.7006),   (29, 30, 0.5075, 0.2585),   (30, 31, 0.9744, 0.963), 
    (31, 32, 0.3105, 0.3619),   (32, 33, 0.341, 0.5302),
]

for i, (from_b, to_b, r, x) in enumerate(line_data):
    network.add(
        "Line", 
        f"Line_{from_b}-{to_b}",
        bus0=f"Bus_{from_b}",
        bus1=f"Bus_{to_b}",
        r=r,
        x=x,
        s_nom=5000
    )

Add the loads. p_set is the active power (kW) and q_set is the reactive power (kVAr)

In [84]:
load_data = [
    (0,0),(100, 60), (90, 40), (120, 80), (60, 30), (60, 20),
    (200, 100), (200, 100), (60, 20), (60, 20), (45, 30),
    (60, 35), (60, 35), (120, 80), (60, 10), (60, 20),
    (60, 20), (90, 40), (90, 40), (90, 40), (90, 40),
    (90, 40), (90, 50), (420, 200), (420, 200), (60, 25),
    (60, 25), (60, 20), (120, 70), (200, 600), (150, 70),
    (210, 100), (60, 40)
]

Apply the time varying profiles

In [85]:
for i, (p, q) in enumerate(load_data):
    real_load_profile = []
    real_load_profile = np.array(load_series) * p

    # load_profile_series = pd.Series(real_load_profile, index=network.snapshots)
    network.add(
        "Load",
        f"Load_bus_{i+1}",
        bus=f"Bus_{i+1}",
        p_set=real_load_profile,
        q_set=q
    )

Add grid, PV and batteries

In [86]:
network.add(
    "Generator",
    "Substation",
    bus="Bus_1",
    p_nom=4000,
    p_min_pu=-2000,
    carrier="gas",
    marginal_cost=price_series
)



for i in range(1, 34):

    # raw_pv_col = pv_cols[i % len(pv_cols)]
    # raw_pv = dataset_day[raw_pv_col].fillna(0)
    # raw_pv_series = raw_pv.values
    # real_pv_profile = raw_pv_series / raw_pv_series.max()
    # # real_pv_profile = real_pv_profile.fillna(0)
    real_pv = (load_data[i-1][0]+10) * pv_series

    network.add(
        "Generator",
        f"PV_bus_{i}",
        bus=f"Bus_{i}",
        p_nom=150,
        p_max_pu=pv_series,
        carrier="solar",
        marginal_cost=0
    )
    network.add(
        "StorageUnit",
        f"Battery_bus_{i}",
        bus=f"Bus_{i}",
        p_nom=20, # Nominal capacity in kW
        max_hours=4, # Energy capacity is p_nom * max_hours = 800 kWh
        carrier="battery",
        efficiency_store=0.9,
        efficiency_dispatch=0.9,
        standing_loss=0.01 # 1% loss per hour
    )

In [87]:
network.optimize()

Index(['Bus_1', 'Bus_2', 'Bus_3', 'Bus_4', 'Bus_5', 'Bus_6', 'Bus_7', 'Bus_8',
       'Bus_9', 'Bus_10', 'Bus_11', 'Bus_12', 'Bus_13', 'Bus_14', 'Bus_15',
       'Bus_16', 'Bus_17', 'Bus_18', 'Bus_19', 'Bus_20', 'Bus_21', 'Bus_22',
       'Bus_23', 'Bus_24', 'Bus_25', 'Bus_26', 'Bus_27', 'Bus_28', 'Bus_29',
       'Bus_30', 'Bus_31', 'Bus_32', 'Bus_33'],
      dtype='str', name='Bus')
Index(['Battery_bus_1', 'Battery_bus_2', 'Battery_bus_3', 'Battery_bus_4',
       'Battery_bus_5', 'Battery_bus_6', 'Battery_bus_7', 'Battery_bus_8',
       'Battery_bus_9', 'Battery_bus_10', 'Battery_bus_11', 'Battery_bus_12',
       'Battery_bus_13', 'Battery_bus_14', 'Battery_bus_15', 'Battery_bus_16',
       'Battery_bus_17', 'Battery_bus_18', 'Battery_bus_19', 'Battery_bus_20',
       'Battery_bus_21', 'Battery_bus_22', 'Battery_bus_23', 'Battery_bus_24',
       'Battery_bus_25', 'Battery_bus_26', 'Battery_bus_27', 'Battery_bus_28',
       'Battery_bus_29', 'Battery_bus_30', 'Battery_bus_31', 'Batter

('ok', 'optimal')

In [ ]:
print(f"cost: {network.objective:.2f}")



cost: 3606999.97


In [89]:
network.generators_t.p

Generator,Substation,PV_bus_1,PV_bus_2,PV_bus_3,PV_bus_4,PV_bus_5,PV_bus_6,PV_bus_7,PV_bus_8,PV_bus_9,...,PV_bus_24,PV_bus_25,PV_bus_26,PV_bus_27,PV_bus_28,PV_bus_29,PV_bus_30,PV_bus_31,PV_bus_32,PV_bus_33
snapshot,,,,,,,,,,,,,,,,,,,,,
2025-07-15 00:00:00,2727.237223,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,...,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000
2025-07-15 01:00:00,2598.617052,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,...,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000
2025-07-15 02:00:00,2519.900394,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,...,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000
2025-07-15 03:00:00,2513.453993,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,...,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000
2025-07-15 04:00:00,3187.552128,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,...,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000,-0.000000
2025-07-15 05:00:00,2630.749329,0.142360,0.142360,0.142360,0.142360,0.142360,0.142360,0.142360,0.142360,0.142360,...,0.142360,0.142360,0.142360,0.142360,0.142360,0.142360,0.142360,0.142360,0.142360,0.142360
2025-07-15 06:00:00,2670.456785,6.428322,6.428322,6.428322,6.428322,6.428322,6.428322,6.428322,6.428322,6.428322,...,6.428322,6.428322,6.428322,6.428322,6.428322,6.428322,6.428322,6.428322,6.428322,6.428322
2025-07-15 07:00:00,1876.934506,25.502479,25.502479,25.502479,25.502479,25.502479,25.502479,25.502479,25.502479,25.502479,...,25.502479,25.502479,25.502479,25.502479,25.502479,25.502479,25.502479,25.502479,25.502479,25.502479
2025-07-15 08:00:00,1641.792292,55.743817,55.743817,55.743817,55.743817,55.743817,55.743817,55.743817,55.743817,55.743817,...,55.743817,55.743817,55.743817,55.743817,55.743817,55.743817,55.743817,55.743817,55.743817,55.743817


In [90]:
network.storage_units_t.p

StorageUnit,Battery_bus_1,Battery_bus_2,Battery_bus_3,Battery_bus_4,Battery_bus_5,Battery_bus_6,Battery_bus_7,Battery_bus_8,Battery_bus_9,Battery_bus_10,...,Battery_bus_24,Battery_bus_25,Battery_bus_26,Battery_bus_27,Battery_bus_28,Battery_bus_29,Battery_bus_30,Battery_bus_31,Battery_bus_32,Battery_bus_33
snapshot,,,,,,,,,,,,,,,,,,,,,
2025-07-15 00:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2025-07-15 01:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2025-07-15 02:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2025-07-15 03:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2025-07-15 04:00:00,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,...,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000,-20.000000
2025-07-15 05:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2025-07-15 06:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2025-07-15 07:00:00,15.718844,15.718844,15.718844,15.718844,15.718844,15.718844,15.718844,15.718844,15.718844,15.718844,...,15.718844,15.718844,15.718844,15.718844,15.718844,15.718844,15.718844,15.718844,15.718844,15.718844
2025-07-15 08:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [91]:
network.loads_t.p
print("a")
load_tot = []
for i in range(24):
    load_0 = network.loads_t.p.Load_bus_2.values[i] 
    + network.loads_t.p.Load_bus_3.values[i] 
    + network.loads_t.p.Load_bus_4.values[i] 
    + network.loads_t.p.Load_bus_5.values[i]
    + network.loads_t.p.Load_bus_6.values[i] 
    + network.loads_t.p.Load_bus_7.values[i] 
    + network.loads_t.p.Load_bus_8.values[i]
    + network.loads_t.p.Load_bus_9.values[i] 
    + network.loads_t.p.Load_bus_10.values[i] 
    + network.loads_t.p.Load_bus_11.values[i]
    + network.loads_t.p.Load_bus_12.values[i] 
    + network.loads_t.p.Load_bus_13.values[i] 
    + network.loads_t.p.Load_bus_14.values[i]
    + network.loads_t.p.Load_bus_15.values[i] 
    + network.loads_t.p.Load_bus_16.values[i] 
    + network.loads_t.p.Load_bus_17.values[i]
    + network.loads_t.p.Load_bus_18.values[i] 
    + network.loads_t.p.Load_bus_19.values[i] 
    + network.loads_t.p.Load_bus_20.values[i]
    + network.loads_t.p.Load_bus_21.values[i] 
    + network.loads_t.p.Load_bus_22.values[i] 
    + network.loads_t.p.Load_bus_23.values[i]
    + network.loads_t.p.Load_bus_24.values[i] 
    + network.loads_t.p.Load_bus_25.values[i] 
    + network.loads_t.p.Load_bus_26.values[i]
    + network.loads_t.p.Load_bus_27.values[i] 
    + network.loads_t.p.Load_bus_28.values[i] 
    + network.loads_t.p.Load_bus_29.values[i]
    + network.loads_t.p.Load_bus_30.values[i] 
    + network.loads_t.p.Load_bus_31.values[i] 
    + network.loads_t.p.Load_bus_32.values[i]
    + network.loads_t.p.Load_bus_33.values[i] 
    print(load_0)

a
73.41149994868714
69.94931498903307
67.83042783465397
67.65690425909624
68.03639644647174
70.94070526202395
77.59330816346434
87.13965452729367
93.71031618293392
96.39601939562993
98.75472253757154
100.0
99.59220605211476
97.67174993482308
95.59810416450208
94.71705251187686
93.35759593117572
92.7166789796323
92.01256113722084
91.41659872756814
88.77326847725692
86.01058001398515
82.55355821334301
77.39208838308346
